# 05 — Independent Continuous Regression Inference Demo

Consumidor independente e read-only dos artifacts finais do Notebook 04. Este demo valida o contrato e produz estimativas contínuas; não treina, não seleciona modelos e não mede performance científica ou validade operacional.

## 1. Independent startup e boundary

In [1]:
from __future__ import annotations

from copy import deepcopy
import importlib
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def discover_project_root() -> Path:
    configured = os.getenv("DATASET_STUDY_ROOT")
    candidates = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        root = candidate.resolve()
        if (root / "pyproject.toml").is_file() and (root / "scripts" / "smoke_predict.py").is_file():
            return root
    raise RuntimeError("Dataset-study project root could not be discovered.")

PROJECT_ROOT = discover_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.finalize_model as finalize_model_module
import scripts.smoke_predict as smoke_predict_module
importlib.reload(finalize_model_module)
importlib.reload(smoke_predict_module)

from scripts.finalize_model import load_and_validate_final_model_handoff, load_and_validate_inference_bundle, sha256_file
from scripts.smoke_predict import (
    InferenceContractError, InferenceInputError, continuous_output_to_frame,
    load_validated_inference_pipeline, normalize_continuous_inference_input,
    predict_continuous_batch, validate_bundle_handoff_alignment,
    validate_inference_readiness, validate_model_artifact_before_load,
)

FINAL_DIR = Path("artifacts/models/concrete-compressive-strength")
HANDOFF_PATH = FINAL_DIR / "final-model-handoff.json"
BUNDLE_PATH = FINAL_DIR / "inference-bundle.json"
FINAL_NAMES = ("final-model-handoff.json", "inference-bundle.json", "final-model-manifest.json", "final-test-evidence.json", "final-pipeline.joblib")

def artifact_snapshot():
    return {name: {"path": (FINAL_DIR / name).as_posix(), "sha256": sha256_file(PROJECT_ROOT / FINAL_DIR / name), "size": (PROJECT_ROOT / FINAL_DIR / name).stat().st_size} for name in FINAL_NAMES}

initial_names = sorted(path.name for path in (PROJECT_ROOT / FINAL_DIR).iterdir())
initial_snapshot = artifact_snapshot()
display(pd.DataFrame(initial_snapshot).T)

,path,sha256,size
final-model-handoff.json,artifacts/models/concrete-compressive-strength...,13fb495eace8cd84689e746b381f1f0ec68a622961e359...,4811
inference-bundle.json,artifacts/models/concrete-compressive-strength...,5445a88ff4ba58798f3214da805375328d6a29406533cb...,5687
final-model-manifest.json,artifacts/models/concrete-compressive-strength...,e42c52d91334ec900c7f8dfc5e28ec9a3d15e5276ad72d...,8385
final-test-evidence.json,artifacts/models/concrete-compressive-strength...,71185852c04cb00b450d00217d2b030cd8eada3774bbdb...,2550
final-pipeline.joblib,artifacts/models/concrete-compressive-strength...,6e6a5a970c6e91b4ae075c48e4cd4a3c21f0b45a9223e3...,561353


## 2. Final handoff, trust e runtime

Os loaders schema-aware são a autoridade para validar o conjunto final. Referências de lineage são apenas metadata; nenhum artifact de preparação ou seleção é aberto. O SHA é verificado antes da desserialização.

In [2]:
handoff = load_and_validate_final_model_handoff(project_root=PROJECT_ROOT, handoff_path=HANDOFF_PATH)
bundle = load_and_validate_inference_bundle(project_root=PROJECT_ROOT, bundle_path=BUNDLE_PATH)
validate_inference_readiness(handoff, bundle)
validate_bundle_handoff_alignment(handoff, bundle)
validated_path = validate_model_artifact_before_load(project_root=PROJECT_ROOT, bundle=bundle, handoff=handoff)
pipeline, loaded_handoff, loaded_bundle, runtime_report = load_validated_inference_pipeline(
    project_root=PROJECT_ROOT, handoff_path=HANDOFF_PATH, bundle_path=BUNDLE_PATH, trusted_source=True
)
summary = {
    "handoff_schema": handoff["schema_version"], "bundle_schema": bundle["schema_version"],
    "dataset_slug": bundle["dataset_slug"], "problem_type": bundle["problem_type"],
    "bundle_sha256": handoff["bundle_reference"]["sha256"],
    "manifest_sha256": handoff["manifest_reference"]["sha256"],
    "model_sha256": bundle["model_artifact_sha256"],
    "inference_demo_ready": bundle["readiness"]["inference_demo_ready"],
    "operational_modeling_ready": bundle["readiness"]["operational_modeling_ready"],
    "operational_validity": bundle["readiness"]["operational_validity"],
    "runtime_load_safe": runtime_report.compatible, "pipeline_fitted": True,
    "steps": list(pipeline.named_steps), "model_identity": pipeline.named_steps["model"].__class__.__name__,
    "feature_order": bundle["feature_order"], "prediction_contract": bundle["prediction_contract"],
}
display(pd.Series(summary, name="validated final consumer state").to_frame())

RuntimeCompatibilityError: Runtime is not safe for model deserialization: python(expected=3.12.13, observed=3.11.15)

## 3. Contrato de entrada e saída

In [ ]:
input_contract = pd.DataFrame([
    {"position": position, "feature": feature, **bundle["input_feature_dtypes"][feature], "required": True, "missing/non-finite behavior": "reject"}
    for position, feature in enumerate(bundle["feature_order"], start=1)
])
display(input_contract)
display(pd.Series(bundle["target_contract"], name="target_contract").to_frame())
display(pd.Series(bundle["prediction_contract"], name="prediction_contract").to_frame())

## 4. Entradas manuais independentes

Os quatro casos abaixo foram escritos manualmente para demonstrar a interface. Não são amostras do dataset, observações reais, ground truth, benchmark, limites científicos ou afirmações de domínio operacional.

In [ ]:
manual_examples = {
    "illustrative_mix_early_age": [300.0, 0.0, 0.0, 190.0, 5.0, 1040.0, 750.0, 7],
    "illustrative_mix_standard": [350.0, 50.0, 0.0, 180.0, 7.0, 1000.0, 780.0, 28],
    "illustrative_mix_slag_fly_ash": [280.0, 100.0, 80.0, 175.0, 9.0, 980.0, 760.0, 56],
    "illustrative_mix_high_cement": [450.0, 0.0, 0.0, 165.0, 11.0, 1020.0, 700.0, 90],
}
demo_input = pd.DataFrame.from_dict(manual_examples, orient="index", columns=bundle["feature_order"])
demo_input.index.name = "example_id"
validated_demo = normalize_continuous_inference_input(demo_input, bundle=bundle)
display(validated_demo)

## 5. Rejeições pedagógicas do contrato

In [ ]:
first = bundle["feature_order"][0]
invalid_inputs = {
    "missing feature": demo_input.drop(columns=[first]),
    "extra feature": demo_input.assign(unexpected=1),
    "wrong order": demo_input.loc[:, list(reversed(bundle["feature_order"]))],
    "non-numeric": demo_input.assign(**{first: "not-a-number"}),
    "NaN": demo_input.assign(**{first: np.nan}),
    "infinity": demo_input.assign(**{first: np.inf}),
}
rejections = []
for case, value in invalid_inputs.items():
    try:
        normalize_continuous_inference_input(value, bundle=bundle)
    except (InferenceInputError, InferenceContractError) as exc:
        rejections.append({"case": case, "rejected": True, "message": str(exc)})
    else:
        raise AssertionError(f"Invalid input accepted: {case}")
display(pd.DataFrame(rejections))

## 6. Predições contínuas e interpretação

Cada prediction estima o target declarado no contrato, na unidade também declarada. Os exemplos não têm ground truth; portanto este notebook não calcula métricas nem mede accuracy. A performance final-test pertence exclusivamente ao Notebook 04. Uma prediction não confirma validade operacional.

In [ ]:
predictions = predict_continuous_batch(pipeline, demo_input, bundle=bundle, runtime_report=runtime_report)
prediction_table = continuous_output_to_frame(predictions, bundle=bundle)
assert np.isfinite(prediction_table["prediction"]).all()
assert prediction_table["unit"].eq(bundle["prediction_contract"]["unit"]).all()
display(prediction_table)

## 7. Repeatability e evidência read-only

A repetição abaixo comprova repeatability de inferência com a mesma batch e o mesmo artifact fitted; não é uma nova avaliação científica.

In [ ]:
repeated = predict_continuous_batch(pipeline, demo_input, bundle=bundle, runtime_report=runtime_report)
deterministic = np.array_equal(predictions.to_numpy(), repeated.to_numpy())
final_snapshot = artifact_snapshot()
final_names = sorted(path.name for path in (PROJECT_ROOT / FINAL_DIR).iterdir())
artifacts_unchanged = initial_snapshot == final_snapshot
no_artifact_created = initial_names == final_names
assert deterministic and artifacts_unchanged and no_artifact_created
read_only_evidence = {"deterministic_inference": deterministic, "artifacts_unchanged": artifacts_unchanged, "no_artifact_created": no_artifact_created, "no_training_operation": True, "no_split_access": True, "readiness_unchanged": loaded_bundle["readiness"] == bundle["readiness"]}
display(pd.Series(read_only_evidence, name="evidence").to_frame())

## 8. Final presentation

In [ ]:
final_presentation = {
    "model_identity": pipeline.named_steps["model"].__class__.__name__,
    "model_sha256_short": bundle["model_artifact_sha256"][:12],
    "feature_count": len(bundle["feature_order"]), "feature_order": bundle["feature_order"],
    "target": bundle["target_contract"]["column"], "prediction_type": bundle["prediction_contract"]["type"],
    "unit": bundle["prediction_contract"]["unit"], "deterministic_inference": deterministic,
    "artifacts_unchanged": artifacts_unchanged, "inference_demo_status": "complete",
    "operational_modeling_ready": bundle["readiness"]["operational_modeling_ready"],
    "operational_validity": bundle["readiness"]["operational_validity"],
}
display(pd.Series(final_presentation, name="Independent inference demo").to_frame())
display(prediction_table)